## Quickstart: Compare runs, choose a model, and deploy it to a REST API

In this quickstart, you will:

- Run a hyperparameter sweep on a training script (PyTorch ANN, replacing Keras/TensorFlow)

- Compare the results of the runs in the MLflow UI

- Choose the best run and register it as a model

- Deploy the model to a REST API

- Build a container image suitable for deployment to a cloud platform

---
### Key changes from the original notebook
| Original (Keras/TF) | This notebook (PyTorch) |
|---|---|
| `keras.Sequential` | `torch.nn.Sequential` |
| `keras.layers.Normalization` | Manual z-score normalization (mean/std) |
| `model.compile()` + `model.fit()` | Custom training loop with `optimizer.step()` |
| `mlflow.tensorflow.log_model` | `mlflow.pytorch.log_model` |
| `keras.optimizers.SGD` | `torch.optim.SGD` |

### Cell 1 – Imports

We replace `keras` with `torch` and `torch.nn`.  
`hyperopt`, `sklearn`, `mlflow`, `numpy`, and `pandas` remain the same.

In [1]:
# ── Standard library & data science ──────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# ── PyTorch (replaces Keras / TensorFlow) ────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ── Hyperparameter optimisation (unchanged) ──────────────────────────
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe

# ── MLflow tracking & model registry ─────────────────────────────────
import mlflow
import mlflow.pytorch          # ← pytorch flavour, NOT mlflow.tensorflow
from mlflow.models import infer_signature

print(f"PyTorch  : {torch.__version__}")
print(f"MLflow   : {mlflow.__version__}")

/workspaces/MLFLOWSTARTER/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/workspaces/MLFLOWSTARTER/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch  : 2.11.0+cu130
MLflow   : 3.10.1


### Cell 2 – Load the Wine-Quality dataset

No change from the original — same CSV, same separator.

In [2]:
data= pd.read_csv("wine-quality.csv",sep=',')
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [3]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [4]:
data.duplicated().sum()

np.int64(937)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


In [6]:
data.nunique()

fixed acidity            68
volatile acidity        125
citric acid              87
residual sugar          310
chlorides               160
free sulfur dioxide     132
total sulfur dioxide    251
density                 890
pH                      103
sulphates                79
alcohol                 103
quality                   7
dtype: int64

In [7]:
data.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000
mean,6.854788,0.278241,0.334192,6.391415,0.045772,35.308085,138.360657,0.994027,3.188267,0.489847,10.514267,5.877909
std,0.843868,0.100795,0.121020,5.072058,0.021848,17.007137,42.498065,0.002991,0.151001,0.114126,1.230621,0.885639
min,3.800000,0.080000,0.000000,0.600000,0.009000,2.000000,9.000000,0.987110,2.720000,0.220000,8.000000,3.000000
25%,6.300000,0.210000,0.270000,1.700000,0.036000,23.000000,108.000000,0.991723,3.090000,0.410000,9.500000,5.000000
50%,6.800000,0.260000,0.320000,5.200000,0.043000,34.000000,134.000000,0.993740,3.180000,0.470000,10.400000,6.000000
75%,7.300000,0.320000,0.390000,9.900000,0.050000,46.000000,167.000000,0.996100,3.280000,0.550000,11.400000,6.000000
max,14.200000,1.100000,1.660000,65.800000,0.346000,289.000000,440.000000,1.038980,3.820000,1.080000,14.200000,9.000000


In [8]:
# ── 1. Outer train / test split (75 / 25) ────────────────────────────
train, test = train_test_split(data, test_size=0.25, random_state=42)

# ── 2. Separate features and target ──────────────────────────────────
train_x = train.drop(["quality"], axis=1).values.astype(np.float32)
train_y = train[["quality"]].values.ravel().astype(np.float32)

test_x  = test.drop(["quality"], axis=1).values.astype(np.float32)
test_y  = test[["quality"]].values.ravel().astype(np.float32)

# ── 3. Inner train / validation split (80 / 20) ───────────────────────
train_x, valid_x, train_y, valid_y = train_test_split(
    train_x, train_y, test_size=0.20, random_state=42
)

# ── 4. MLflow signature (same as original, uses NumPy arrays) ─────────
#    infer_signature expects NumPy/pandas, so pass the raw arrays
signature = infer_signature(train_x, train_y)

print(f"train_x  : {train_x.shape}   train_y  : {train_y.shape}")
print(f"valid_x  : {valid_x.shape}   valid_y  : {valid_y.shape}")
print(f"test_x   : {test_x.shape}    test_y   : {test_y.shape}")

train_x  : (2938, 11)   train_y  : (2938,)
valid_x  : (735, 11)   valid_y  : (735,)
test_x   : (1225, 11)    test_y   : (1225,)


### Cell 4 – Build PyTorch DataLoaders
 
PyTorch requires explicit **`TensorDataset` → `DataLoader`** wrappers.

> **`TensorDataset`** pairs features and labels.  
> **`DataLoader`** yields shuffled mini-batches of a given `batch_size`.

In [9]:
BATCH_SIZE = 64

def make_loader(X: np.ndarray, y: np.ndarray, shuffle: bool = False) -> DataLoader:
    """Wrap NumPy arrays in a PyTorch DataLoader."""
    X_t = torch.from_numpy(X)           # shape: (N, 11)
    y_t = torch.from_numpy(y).unsqueeze(1)  # shape: (N, 1) — needed for MSELoss
    return DataLoader(TensorDataset(X_t, y_t), batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(train_x, train_y, shuffle=True)
valid_loader = make_loader(valid_x, valid_y)
test_loader  = make_loader(test_x,  test_y)

### Cell 5 – Define the ANN model

The architecture mirrors the original Keras model:

```
Input(11)  →  Normalization  →  Dense(64, relu)  →  Dense(1)
```

```
x_norm = (x - mean) / sqrt(variance + ε)
```

We implement this as a custom `Normalize` module so it is saved
inside the model (important for correct MLflow serialisation).

In [10]:
class Normalize(nn.Module):
     
     """
     Mirrors Keras' Normalization layer: z-score using training-set
     mean and variance (NOT std — same as Keras default).
     """
     def __init__(self,mean:np.ndarray,var:np.ndarray):
          super().__init__()
          # Register as buffers so they move with .to(device) and are
          # saved with state_dict() / MLflow pickle.
          self.register_buffer("mean", torch.tensor(mean,dtype= torch.float32))
          self.register_buffer("std", torch.tensor(np.sqrt(var+1e-7), dtype=torch.float32))
     def forward(self,x:torch.Tensor)->torch.Tensor:
          return (x - self.mean) / self.std


register_buffer = "This tensor is part of my model.
                   Move it, save it, restore it — automatically."

In [11]:
def build_model(train_x: np.ndarray)-> nn.Sequential:
    """
    Build a 3-layer ANN equivalent to the Keras model:
      Normalization → Linear(11→64, ReLU) → Linear(64→1)
    """
    mean= np.mean(train_x, axis=0)
    var= np.var(train_x, axis=0)
    n_features = train_x.shape[1]

    model= nn.Sequential(
        Normalize(mean,var),
        nn.Linear(n_features,64),
        nn.ReLU(),
        nn.Linear(64,1),
    )
    return model

# Quick sanity-check
_demo= build_model(train_x)
print(_demo)
print("output shape:", _demo(torch.from_numpy(train_x[:4])).shape)

Sequential(
  (0): Normalize()
  (1): Linear(in_features=11, out_features=64, bias=True)
  (2): ReLU()
  (3): Linear(in_features=64, out_features=1, bias=True)
)
output shape: torch.Size([4, 1])


why should I use Function and CLass?

    Function  → same STEPS needed more than once
    
    Class     → same STEPS + need to REMEMBER something between calls

### Cell 6 – Training utilities

Keras hides the epoch loop inside `model.fit()`. In PyTorch we write it
explicitly — this gives full control over logging, gradient clipping, etc.

```
for batch in train_loader:
    optimizer.zero_grad()
    loss = criterion(model(X), y)
    loss.backward()
    optimizer.step()
```

In [12]:
def run_epoch(
        model: nn.Module,
        loader: DataLoader,
        criterion: nn.Module,
        optimizer= None, # None-> eval mode(no gradient)
) -> float:
    
     """
    Run one forward pass over `loader`.
    If `optimizer` is provided, also run backward + update (training mode).
    Returns the root-mean-squared error over the full loader.
    """
     is_train= optimizer is not None
     model.train(is_train)

     total_sq_err, total_n= 0.0,0
     with torch.set_grad_enabled(is_train):
          for X_batch, y_batch in loader:
               preds= model(X_batch)  # forward pass
               loss= criterion(preds, y_batch) # MSE loss

               if is_train:
                    optimizer.zero_grad() # clear old gradients
                    loss.backward()   #backprop
                    optimizer.step()  # weight update

                # accumulate squared error for RMSE calculation
               total_sq_err= total_sq_err + loss.item() * len(y_batch)
               total_n += len(y_batch)
               
     rmse = (total_sq_err / total_n) ** 0.5
     return rmse



### Cell 7 – `train_model`: the MLflow-tracked training function

This is the PyTorch equivalent of the original `train_model` that used
`model.fit()` + `mlflow.tensorflow.log_model`.

Key replacements:

| Original | PyTorch equivalent |
|---|---|
| `keras.optimizers.SGD(lr, momentum)` | `torch.optim.SGD(params, lr, momentum)` |
| `model.fit(...)` | Custom `run_epoch()` loop |
| `model.evaluate(...)` | `run_epoch(..., optimizer=None)` |
| `mlflow.tensorflow.log_model` | `mlflow.pytorch.log_model` |

In [14]:
def train_model(params,epochs,train_loader,valid_loader):

     
    """
        Build, train, and evaluate a PyTorch ANN.
        Logs params, metrics, and the model artifact to MLflow.
        Returns a dict compatible with Hyperopt (loss = eval_rmse).
        """
     # model & optimizer
    model= build_model(train_x)
    criterion= nn.MSELoss()
    optimizer= torch.optim.SGD(
          model.parameters(),
          lr= params["lr"],
          momentum= params["momentum"]
     )
    # MLFLOW nested run(one run per hyperopt trial)
    with mlflow.start_run(nested= True):
        #  training loop(replaces model.fit)
        for epoch in range(1, epochs+1):
            train_rmse= run_epoch(model,train_loader, criterion,optimizer)
            val_rmse= run_epoch(model, valid_loader, criterion) # eval model
            print(f"  Epoch {epoch}/{epochs} — "
                  f"train RMSE: {train_rmse:.4f}  val RMSE: {val_rmse:.4f}")
            
        # final validation RMSE(replaces model.evaluate)
        eval_rmse= run_epoch(model,valid_loader,criterion)
        # ── Log params & metrics to MLflow
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse",eval_rmse)

        # ── Log the PyTorch model artifact ────────────────────────────
        #    mlflow.pytorch.log_model replaces mlflow.tensorflow.log_model
        mlflow.pytorch.log_model(
            model,
            artifact_path="model",
            signature= signature
        )
    return {"loss":eval_rmse, "status": STATUS_OK,"model":model}




### Cell 8 – Hyperopt objective and search space

In [16]:
def objective(params):
    """Hyperopt calls this once per trial"""
    return train_model(
        params,
        epochs=3,
        train_loader=train_loader,
        valid_loader= valid_loader,
    )

space= {
    "lr": hp.loguniform("lr", np.log(1e-56), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0,)
}

### Cell 9 – Run the hyperparameter sweep with MLflow

The outer `mlflow.start_run()` is the **parent run**.  
Each Hyperopt trial creates a **nested child run** via `mlflow.start_run(nested=True)`.  
After the sweep, we pick the best child run and log its parameters and model on the parent.

In [ ]:
mlflow.set_experiment("wine-quality-pytorch")

with mlflow.start_run():
    # hyperparameter search
    trials= Trials()
    best= fmin(
        fn= objective,
        space= space,
        algo= tpe.suggest,
        max_evals= 4,
        trials= trials,
    )

    # Retrieve the best trial
    best_run= sorted(trials.results,key=lambda x:x["loss"])[0]

    # log best params + metric + model on the parent run
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.pytorch.log_model(
        best_run["model"],
        artifact_path= "model",
        signature= signature,
    )

    print(f"Best parameters : {best}")
    print(f"Best eval RMSE  : {best_run['loss']:.4f}")
